<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/sandbox/notebooks/demo_corer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Core Reranker
vLLM-Hook is an extensible framework that aims to allow selective access to model internals during the inference. 
As a demonstration of that, in this notebook, we show how vLLM-Hook enables *Core Reranker* for document relevance scoring. 

**Paper**: [Contrastive Retrieval Heads Improve Attention-Based Re-Ranking](https://arxiv.org/abs/2510.02219).<br />
**Authors**: Linh Tran, Yulong Li, Radu Florian, Wei Sun <br />
**"TL;DR"**: Core reranker is an attention-based reranker that leverage attention weights from selected transformer heads to produce document relevance scores.


### Installation
If running this from a new environment, please use the cell below to install `vllm_hook_plugins`. Update the path/command to match your environment.<br />
The following block is not necessary if running this notebook from an environment where the package has already been installed.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/tburleyinfo/vLLM-Hook.git"
REPO_BRANCH = "sandbox"
REPO_NAME = "vLLM-Hook"

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()

if IN_COLAB:
    REPO_ROOT = NOTEBOOK_DIR / REPO_NAME
    expected_remote = REPO_URL.removesuffix('.git')
    if not REPO_ROOT.exists():
        print(f"Colab detected. Cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_ROOT} ...")
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"Colab detected. Reusing existing clone at {REPO_ROOT}")
        origin_url = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix('.git')
        if origin_url != expected_remote:
            print(f"Remote mismatch ({origin_url}); replacing clone with {expected_remote}")
            shutil.rmtree(REPO_ROOT)
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        else:
            print("Refreshing existing clone ...")
            subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Changed working directory to {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"

print("Colab      :", IN_COLAB)
print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Repo branch :", REPO_BRANCH)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if IN_COLAB:
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "This notebook requires a Colab GPU runtime with CUDA available. "
            "In Colab, use Runtime > Change runtime type > T4 GPU (or another GPU), then rerun from a fresh runtime."
        )

if not PKG_DIR.exists():
    raise FileNotFoundError(f"Plugin directory not found: {PKG_DIR}")

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("Colab was detected but git is unavailable in the runtime.")

if REQ_FILE.exists():
    req_cmd = [sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)]
    print("Running:", " ".join(req_cmd))
    subprocess.run(req_cmd, check=True)
else:
    print("Warning: requirement.txt not found at", REQ_FILE)

# Colab commonly ships TensorFlow/Keras bits that still expect the older
# protobuf MessageFactory.GetPrototype API. Install this after vllm so the
# resolver does not immediately replace it with a newer protobuf build.
protobuf_cmd = [sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"]
print("Running:", " ".join(protobuf_cmd))
subprocess.run(protobuf_cmd, check=True)

pkg_cmd = [sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)]
print("Running:", " ".join(pkg_cmd))
subprocess.run(pkg_cmd, check=True)
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
import importlib
importlib.invalidate_caches()


### Importing the Hook-Enabled LLM
The plugin provides its own LLM wrapper that behaves like vllm.LLM (`from vllm import LLM`) but adds support for hooks and instrumentation.
We import it here:

In [ ]:
from vllm_hook_plugins import HookLLM

### Environment & multiprocessing setup

In [ ]:
import io
import os
import multiprocessing as mp
import sys
import torch
from typing import List

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)
    os.makedirs("/content/.cache/vllm-hook", exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
    print("Colab detected: using fork start method and disabled V1 multiprocessing")
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


### Helper functions that give the instruction range
As Core Reranker needs to locate the candidate passages and the user query in the prompt, below is a helper function that gives the data range with texts.<br />
Check [Core Reranker](https://arxiv.org/pdf/2510.02219) for more details.

In [ ]:
def apply_chat_template_and_get_ranges(tokenizer, model_name: str, query: str, documents: List[str]):
    # setup prompts
    off_set = 0
    if 'granite' in model_name.lower():
        prompt_prefix = '<|start_of_role|>user<|end_of_role|>'
        prompt_suffix = '<|end_of_text|><|start_of_role|>assistant<|end_of_role|>'
    elif 'llama' in model_name.lower():
        prompt_prefix = '<|start_header_id|>user<|end_header_id|>'
        prompt_suffix = '<|eot_id|><|start_header_id|>assistant<|end_header_id|>'
    elif 'mistral' in model_name.lower():
        prompt_prefix = '[INST]'
        prompt_suffix = '[/INST]'
        off_set = 1
    elif 'phi' in model_name.lower():
        prompt_prefix = '<|im_start|>user<|im_sep|>'
        prompt_suffix = '<|im_end|><|im_start|>assistant<|im_sep|>'
    retrieval_instruction = ' Here are some paragraphs:\n\n'
    retrieval_instruction_late = 'Please find information that are relevant to the following query in the paragraphs above.\n\nQuery: '
    
    doc_span = []
    query_start_idx = None
    query_end_idx = None

    llm_prompt = prompt_prefix + retrieval_instruction

    for i, doc in enumerate(documents):

        llm_prompt += f'[document {i+1}]'
        start_len = len(tokenizer(llm_prompt).input_ids)

        llm_prompt += ' ' + " ".join(doc)
        end_len = len(tokenizer(llm_prompt).input_ids) - off_set

        doc_span.append((start_len, end_len))
        llm_prompt += '\n\n'

    start_len = len(tokenizer(llm_prompt).input_ids)

    llm_prompt += retrieval_instruction_late
    after_retrieval_instruction_late = len(tokenizer(llm_prompt).input_ids) - off_set

    llm_prompt += f'{query.strip()}'
    end_len = len(tokenizer(llm_prompt).input_ids) - off_set
    llm_prompt += prompt_suffix

    query_start_idx = start_len
    query_end_idx = end_len

    return llm_prompt, (doc_span, query_start_idx, after_retrieval_instruction_late, query_end_idx)

### Initialize `HookLLM`
Before we create the LLM instance, we need to specify the model and data type:

In [ ]:
cache_dir = '/content/.cache/vllm-hook' if IN_COLAB else os.path.expanduser('~/.cache/vllm-hook')
model = 'mistralai/Mistral-7B-Instruct-v0.3' 
    
dtype_map = {
    'mistralai/Mistral-7B-Instruct-v0.3': torch.float16,
}


We also need to provide a config file that specifies the important heads we want to track. <br />
For Core Reranker, this config file can be obtained from [head_detection.py](https://github.com/linhhtran/CoRe-Reranking/blob/main/experiments/head_detection.py). 

In [ ]:
import json
from pathlib import Path

json_path = Path("../model_configs/core_reranker/Mistral-7B-Instruct-v0.3.json")  # adjust path

with open(json_path, "r") as f:
    config = json.load(f)

# print(config)

Inside `probe_hook_qk` and `core_reranker` we defined the desired behavior during model inference and after the model inference: 
- `workers/probe_hookqk_worker.py` defines that we need `q` (query) and `k` (key) to be saved during forward passes
- `analyzers/core_reranker_analyzer.py` calculates the passage relevance score and the final ranking of passages

Now, we initialize the llm:

In [ ]:
llm = HookLLM(
    model=model,
    worker_name="probe_hook_qk",
    analyzer_name="core_reranker",
    config_file=json_path,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    trust_remote_code=True,
    dtype=dtype_map[model],
    enable_prefix_caching=True,
    enable_hook=True
)

### Test case
In the following, we show a test case with seven candidate passages and a user query.

In [ ]:
case = {
        "query": "Which came first, the invention of the telephone or the light bulb?",
        "documents": [
            [
            "Alexander Graham Bell is credited with inventing the first practical telephone.",
            " He was awarded the U.S. patent for the invention of the telephone on March 7, 1876.",
            " The first successful demonstration of the telephone took place shortly thereafter, when Bell famously called his assistant, saying, 'Mr. Watson, come here, I want to see you.'",
            " Bell’s invention revolutionized communication by allowing people to talk to each other over long distances."
            ],
            [
            "Thomas Edison is widely known for inventing the first commercially practical incandescent light bulb.",
            " Although he did not invent the concept of the light bulb itself, Edison developed a version that was safe, affordable, and long-lasting.",
            " His patent for the electric light bulb was filed in 1879, three years after Bell’s telephone patent.",
            " Edison's innovation led to widespread use of electric lighting and helped usher in the modern electrical age."
            ],
            [
            "Before Edison, several inventors worked on early versions of the light bulb.",
            " Sir Humphry Davy created the first electric arc lamp in the early 1800s, and later inventors like Joseph Swan in Britain improved upon the design.",
            " However, these early bulbs were inefficient or burned out quickly, and it was Edison who perfected the design for everyday use."
            ],
            [
            "The telephone was invented before the practical light bulb.",
            " Bell’s patent for the telephone was issued in 1876, while Edison’s patent for the light bulb was filed in 1879.",
            " Thus, the telephone came first."
            ],
            [
            "Both the telephone and the light bulb are considered groundbreaking inventions of the late 19th century.",
            " The telephone transformed communication, while the light bulb transformed how people lived and worked at night.",
            " Together, they symbolize the rapid technological progress of that era."
            ],
            [
            "Edison and Bell were contemporaries and pioneers of the Second Industrial Revolution.",
            " Their inventions marked major milestones in human history, driving the growth of telecommunications and electrical infrastructure."
            ],
            [
            "In summary, the telephone was invented in 1876 and the light bulb in 1879.",
            " Therefore, the invention of the telephone came first."
            ]
        ]
    }

Next, we apply chat template and obtain the input range using the helper function defined above.<br />
Specifically, as core reranker relies on the aggregated attentions from the user query to each passage, it needs a reference attention baseline for each passage. The authors swap the user query with `'N/A'` and treat the resulting aggregated attention as the normalizing factor for each passage.

In [ ]:
query = case["query"]
documents = case["documents"]
        
# Apply chat template and get ranges
query_text, query_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, query, documents)
na_text, na_spec = apply_chat_template_and_get_ranges(llm.tokenizer, model, 'N/A', documents)

Finally, we perform the model inference:

In [ ]:
llm.generate(query_text, temperature=0.1, max_tokens=1)
llm.generate(na_text, cleanup=False, temperature=0.1, max_tokens=1)

During the model inference in the previous step, vLLM-Hook has automatically saved selected queries and keys. Now, we can directly call the analyzer to get the passage relevance score and the final ranking of passages:

In [ ]:

stats = llm.analyze(analyzer_spec={'query_spec': query_spec, 'na_spec': na_spec})

Finally we can print out the results as follows:

In [ ]:
print(f"Sorted document IDs and scores by CoRe-Reranking: {stats['ranking']}: {stats['scores']}")